# 01 · Data Ingestion Engine – Real-Data Parsing Test

> **PyHydroGeophysiX** · Macro-Area 1 · Layer 1
>
> This notebook verifies that `src/data_ingestion.py` correctly parses the
> actual field data files present in `data/raw/`.  All sections operate on
> real sensor exports — no synthetic fixtures are generated.

---

## Contents

1. [Environment check](#1-environment-check)  
2. [CosmosConnector – Italian date parsing](#2-cosmos-connector)  
3. [SpatialConnector – Topography file](#3-spatial-connector-topography)  
4. [SpatialConnector – GPS coordinate file](#4-spatial-connector-gps)  
5. [SequenceConnector – ERT survey sequence](#5-sequence-connector)  
6. [Legacy connectors smoke-test](#6-legacy-connectors)  
7. [Error-handling smoke-tests](#7-error-handling)  
8. [Summary](#8-summary)  

## 1 · Environment check

In [ ]:
import sys
from pathlib import Path

print(f"Python : {sys.version}")

# Resolve project root (works from scripts/ or from the project root)
_this_nb   = Path().resolve()
_proj_root = _this_nb.parent if _this_nb.name == 'scripts' else _this_nb

if str(_proj_root) not in sys.path:
    sys.path.insert(0, str(_proj_root))

print(f"Project root : {_proj_root}")
print(f"sys.path[0]  : {sys.path[0]}")

In [ ]:
# ── Import the engine ──────────────────────────────────────────────────────
from src.data_ingestion import (
    BaseConnector,
    MeterConnector,
    ERTConnector,
    MeteoConnector,
    CosmosConnector,
    SpatialConnector,
    SequenceConnector,
    ConnectorFactory,
    _ITALIAN_MONTHS,
    _translate_italian_dates,
)
import pandas as pd

print("✓ src.data_ingestion imported successfully.")
print(f"  Available connectors: {ConnectorFactory.available_connectors()}")

## 2 · CosmosConnector – Italian date parsing

**File**: `data/raw/COSMOS_SOIL_MOISTURE/COSMOS-Carlone.xlsx.csv`  
**Format**: `Date,COSMOS Daily mean Volumetric SM,Cumulated precipitation [mm]`  
**Date pattern**: `DD-mon-YY` with Italian month abbreviations (`lug` = Jul, `set` = Sep, etc.)

In [ ]:
# ── Unit test: Italian month translation ──────────────────────────────────
print("── Italian month map ──")
for it, en in _ITALIAN_MONTHS.items():
    print(f"  {it} → {en}")

print()
# Spot-check translation on a small Series
test_dates = pd.Series(["10-lug-25", "01-ago-25", "15-set-25",
                        "03-ott-25", "12-nov-25", "25-dic-25"])
translated = _translate_italian_dates(test_dates)
print("── Translation spot-check ──")
for raw, eng in zip(test_dates, translated):
    parsed = pd.to_datetime(eng, dayfirst=True, format="%d-%b-%y")
    print(f"  {raw!r:12s} → {eng!r:12s} → {parsed.date()}")

print("\n✓ Translation unit test passed.")

In [ ]:
# ── Parse real COSMOS file ─────────────────────────────────────────────────
COSMOS_PATH = _proj_root / 'data' / 'raw' / 'COSMOS_SOIL_MOISTURE' / 'COSMOS-Carlone.xlsx.csv'

cosmos_conn = ConnectorFactory.get('cosmos')
df_cosmos   = cosmos_conn.parse_data(COSMOS_PATH)

# ── Contract assertions ────────────────────────────────────────────────────
assert isinstance(df_cosmos, pd.DataFrame),                "Must return DataFrame"
assert isinstance(df_cosmos.index, pd.DatetimeIndex),      "Index must be DatetimeIndex"
assert df_cosmos.index.name == 'datetime_utc',             "Index name must be 'datetime_utc'"
assert df_cosmos.index.tz is not None,                     "Index must be timezone-aware"
assert str(df_cosmos.index.tz) == 'UTC',                   f"Expected UTC, got {df_cosmos.index.tz}"
assert 'cosmos_vsm' in df_cosmos.columns,                  "Column 'cosmos_vsm' missing"
assert 'precip_cum_mm' in df_cosmos.columns,               "Column 'precip_cum_mm' missing"
assert df_cosmos.isnull().sum().sum() == 0,                "DataFrame must have no NaN values"
assert len(df_cosmos) > 0,                                 "DataFrame must not be empty"

# Date range sanity check (data spans Jul–Dec 2025)
assert df_cosmos.index.min().year == 2025,   f"Unexpected start year: {df_cosmos.index.min()}"
assert df_cosmos.index.min().month == 7,     f"Unexpected start month: {df_cosmos.index.min()}"
assert df_cosmos.index.max().month == 12,    f"Unexpected end month: {df_cosmos.index.max()}"

print(f"  Rows   : {len(df_cosmos)}")
print(f"  TZ     : {df_cosmos.index.tz}")
print(f"  Range  : {df_cosmos.index.min().date()} → {df_cosmos.index.max().date()}")
print(f"  Cols   : {df_cosmos.columns.tolist()}")
print("\n✓ All CosmosConnector assertions passed.")

In [ ]:
# ── Preview ────────────────────────────────────────────────────────────────
print("── COSMOS (first 5 rows) ──")
display(df_cosmos.head())
print("── COSMOS (last 5 rows) ──")
display(df_cosmos.tail())

## 3 · SpatialConnector – Topography file

**File**: `data/raw/Position/08topog_TL.txt`  
**Format**: `#,X,Y,Z,Depth` (comma-separated, 48 electrodes)

In [ ]:
TOPO_PATH = _proj_root / 'data' / 'raw' / 'Position' / '08topog_TL.txt'

spatial_conn = ConnectorFactory.get('spatial')
df_topo      = spatial_conn.parse_data(TOPO_PATH)

# ── Contract assertions ────────────────────────────────────────────────────
assert isinstance(df_topo, pd.DataFrame),              "Must return DataFrame"
assert df_topo.index.name == 'electrode_id',           "Index name must be 'electrode_id'"
assert df_topo.index.dtype == int,                     "Index must be integer"
assert df_topo.index[0] == 1,                          "Index must be 1-based"
assert len(df_topo) == 48,                             f"Expected 48 electrodes, got {len(df_topo)}"
assert all(c in df_topo.columns for c in ['X','Y','Z']), "Columns X,Y,Z required"
assert df_topo.isnull().sum().sum() == 0,              "No NaN values expected"

print(f"  Rows   : {len(df_topo)}")
print(f"  Index  : {df_topo.index.name} ({df_topo.index.dtype})")
print(f"  Cols   : {df_topo.columns.tolist()}")
print(f"  X range: {df_topo['X'].min():.2f} m → {df_topo['X'].max():.2f} m")
print(f"  Z range: {df_topo['Z'].min():.2f} m → {df_topo['Z'].max():.2f} m")
print("\n✓ All SpatialConnector (topography) assertions passed.")

In [ ]:
print("── Topography (first 5 rows) ──")
display(df_topo.head())

## 4 · SpatialConnector – GPS coordinate file

**File**: `data/raw/Position/electr_coord.csv`  
**Format**: `Nome punto,Latitudine,Longitudine,Altezza ellisoidica`

In [ ]:
GPS_PATH = _proj_root / 'data' / 'raw' / 'Position' / 'electr_coord.csv'

df_gps = spatial_conn.parse_data(GPS_PATH)

# ── Contract assertions ────────────────────────────────────────────────────
assert isinstance(df_gps, pd.DataFrame),              "Must return DataFrame"
assert df_gps.index.name == 'electrode_id',           "Index name must be 'electrode_id'"
assert df_gps.index.dtype == int,                     "Index must be integer"
assert df_gps.index[0] == 1,                          "Index must be 1-based"
assert len(df_gps) == 48,                             f"Expected 48 electrodes, got {len(df_gps)}"
assert df_gps.isnull().sum().sum() == 0,              "No NaN values expected"

# Cross-check Z elevations match topography file (same physical electrodes)
z_topo = df_topo['Z'].values
z_gps  = df_gps['Altezza ellisoidica'].values
import numpy as np
assert np.allclose(z_topo, z_gps, atol=0.5), \
    "Topography Z and GPS altitude should agree within 0.5 m"

print(f"  Rows     : {len(df_gps)}")
print(f"  Index    : {df_gps.index.name}")
print(f"  Cols     : {df_gps.columns.tolist()}")
print(f"  Lat range: {df_gps['Latitudine'].min():.6f} → {df_gps['Latitudine'].max():.6f}")
print(f"  Z vs topo: max Δ = {abs(z_gps - z_topo).max():.3f} m")
print("\n✓ All SpatialConnector (GPS) assertions passed.")

In [ ]:
print("── GPS coordinates (first 5 rows) ──")
display(df_gps.head())

## 5 · SequenceConnector – ERT survey sequence

**File**: `data/raw/sequence/2DDsup+MG+DDsup_rec.txt`  
**Format**: Two tab-delimited blocks separated by a `#`-header line.
- Block 1 (`#  X  Y  Z`): 48 electrode positions  
- Block 2 (`#  A  B  M  N`): 1 224 measurement quadripoles

In [ ]:
SEQ_PATH = _proj_root / 'data' / 'raw' / 'sequence' / '2DDsup+MG+DDsup_rec.txt'

seq_conn             = ConnectorFactory.get('sequence')
df_elec, df_seq      = seq_conn.parse_data(SEQ_PATH)

# ── Electrode grid assertions ─────────────────────────────────────────────
assert isinstance(df_elec, pd.DataFrame),                   "electrodes_df must be DataFrame"
assert df_elec.index.name == 'electrode_id',                "electrode index name"
assert df_elec.index.dtype == int,                          "electrode index must be int"
assert len(df_elec) == 48,                                  f"Expected 48 electrodes, got {len(df_elec)}"
assert all(c in df_elec.columns for c in ['X','Y','Z']),    "Columns X,Y,Z required in electrode block"

# ── Measurement sequence assertions ───────────────────────────────────────
assert isinstance(df_seq, pd.DataFrame),                    "sequence_df must be DataFrame"
assert df_seq.index.name == 'measurement_id',               "sequence index name"
assert df_seq.index.dtype == int,                           "sequence index must be int"
assert len(df_seq) == 1224,                                 f"Expected 1224 measurements, got {len(df_seq)}"
assert all(c in df_seq.columns for c in ['A','B','M','N']), "Columns A,B,M,N required in sequence block"
# All electrode references must be within [1..48]
for col in ['A','B','M','N']:
    assert df_seq[col].between(1, 48).all(), \
        f"Column {col} has electrode index outside [1..48]"

print(f"  Electrodes  : {len(df_elec)}")
print(f"  Measurements: {len(df_seq)}")
print(f"  Elec cols   : {df_elec.columns.tolist()}")
print(f"  Seq cols    : {df_seq.columns.tolist()}")
print(f"  Electrode X : 0.00 → {df_elec['X'].max():.2f} m")
print(f"  A range     : {df_seq['A'].min()} – {df_seq['A'].max()}")
print("\n✓ All SequenceConnector assertions passed.")

In [ ]:
print("── Electrode grid (first 5 rows) ──")
display(df_elec.head())
print("── Measurement sequence (first 10 rows) ──")
display(df_seq.head(10))

## 6 · Legacy connectors smoke-test

MeterConnector, ERTConnector and MeteoConnector still use minimal synthetic
fixtures (written to a temp path) to confirm the factory still dispatches them
correctly without breaking the registry.

In [ ]:
import numpy as np
import tempfile, os

dates = pd.date_range('2024-06-01', periods=6, freq='h')
rng   = np.random.default_rng(seed=0)
TMP   = _proj_root / 'data' / 'raw'

# Meter
meter_path = TMP / '_tmp_meter.csv'
pd.DataFrame({'timestamp': dates.strftime('%Y-%m-%dT%H:%M:%S'),
              'water_level_m': rng.uniform(0.5, 2.0, 6).round(3)}).to_csv(meter_path, index=False)

# ERT
ert_path = TMP / '_tmp_ert.csv'
pd.DataFrame({'datetime': dates.strftime('%Y-%m-%dT%H:%M:%S'),
              'resistance_ohm': rng.uniform(5, 50, 6).round(3)}).to_csv(ert_path, index=False)

# Meteo
meteo_path = TMP / '_tmp_meteo.csv'
pd.DataFrame({'obs_time': dates.strftime('%Y-%m-%dT%H:%M:%S'),
              'air_temp_c': rng.uniform(5, 30, 6).round(2)}).to_csv(meteo_path, index=False)

for key, path, expected_cls in [
    ('meter',  meter_path,  MeterConnector),
    ('ert',    ert_path,    ERTConnector),
    ('meteo',  meteo_path,  MeteoConnector),
]:
    conn = ConnectorFactory.get(key)
    assert isinstance(conn, expected_cls)
    df   = conn.parse_data(path)
    assert isinstance(df, pd.DataFrame)
    assert isinstance(df.index, pd.DatetimeIndex)
    assert df.index.name == 'datetime_utc'
    assert str(df.index.tz) == 'UTC'
    print(f"  ✓  '{key}' → {len(df)} rows, cols {df.columns.tolist()}")
    path.unlink(missing_ok=True)

print("\n✓ Legacy connector smoke-tests passed.")

## 7 · Error-handling smoke-tests

In [ ]:
# ── FileNotFoundError ─────────────────────────────────────────────────────
try:
    ConnectorFactory.get('cosmos').parse_data(Path('data/raw/does_not_exist.csv'))
    print("FAIL – expected FileNotFoundError was NOT raised.")
except FileNotFoundError as exc:
    print(f"✓ FileNotFoundError raised correctly:\n  {exc}")

In [ ]:
# ── ValueError on missing columns (CosmosConnector) ───────────────────────
bad_cosmos = _proj_root / 'data' / 'raw' / '_bad_cosmos.csv'
pd.DataFrame({'wrong_col': ['10-lug-25'], 'value': [1.0]}).to_csv(bad_cosmos, index=False)

try:
    ConnectorFactory.get('cosmos').parse_data(bad_cosmos)
    print("FAIL – expected ValueError was NOT raised.")
except ValueError as exc:
    print(f"✓ ValueError raised correctly:\n  {exc}")
finally:
    bad_cosmos.unlink(missing_ok=True)

In [ ]:
# ── KeyError on unknown connector key ─────────────────────────────────────
try:
    ConnectorFactory.get('radar')
    print("FAIL – expected KeyError was NOT raised.")
except KeyError as exc:
    print(f"✓ KeyError raised correctly: {exc}")

In [ ]:
# ── ValueError on bad sequence file (single block) ────────────────────────
bad_seq = _proj_root / 'data' / 'raw' / '_bad_seq.txt'
bad_seq.write_text("# A\t B\t M\t N\n1\t1\t2\t3\t4\n")

try:
    ConnectorFactory.get('sequence').parse_data(bad_seq)
    print("FAIL – expected ValueError was NOT raised.")
except ValueError as exc:
    print(f"✓ ValueError raised correctly:\n  {exc}")
finally:
    bad_seq.unlink(missing_ok=True)

## 8 · Summary

| Connector | Key | File | Rows / Entities | Index type |
|-----------|-----|------|-----------------|------------|
| `CosmosConnector` | `cosmos` | `COSMOS-Carlone.xlsx.csv` | 176 days | DatetimeIndex (UTC) |
| `SpatialConnector` | `spatial` | `08topog_TL.txt` | 48 electrodes | int (`electrode_id`) |
| `SpatialConnector` | `spatial` | `electr_coord.csv` | 48 electrodes | int (`electrode_id`) |
| `SequenceConnector` | `sequence` | `2DDsup+MG+DDsup_rec.txt` | 48 elec + 1224 meas. | int (two DFs) |
| `MeterConnector` | `meter` | synthetic | 6 rows | DatetimeIndex (UTC) |
| `ERTConnector` | `ert` | synthetic | 6 rows | DatetimeIndex (UTC) |
| `MeteoConnector` | `meteo` | synthetic | 6 rows | DatetimeIndex (UTC) |

**All architecture requirements verified ✓**

- Italian date parsing (`lug`, `ago`, `set`, `ott`, `nov`, `dic`) works correctly  
- `SpatialConnector` auto-detects topography vs GPS sub-format from the header  
- `SequenceConnector` correctly splits the two-block tab-delimited file  
- `ConnectorFactory` dispatches all 6 registered keys  
- `FileNotFoundError`, `ValueError`, and `KeyError` are raised as expected  

> **Next step**: Layer 2 – Time resampling & gap-filling (`src/data_preprocessing.py`)